In [1]:
# Cell 1: Imports & configuration
import os
import json
import datetime
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, text

# --- Paths ---
home = Path.home()
one_drive = home / "OneDrive"
LYRICS_FOLDER = one_drive / "A4" / "Data" / "Videos" / "AWS"
ADDITIONAL_OUTPUT_FOLDER = LYRICS_FOLDER.parent

# --- DB connection ---
DB_URL = 'mysql+pymysql://root@localhost:3306/music_development'

# --- Output file names ---
SONGS_WITH_IDS_FILE   = 'songs-with-ids.json'
SONGS_MISSING_IDS_FILE = 'songs-missing-youtube-ids.json'

print(f"LYRICS_FOLDER          : {LYRICS_FOLDER}")
print(f"ADDITIONAL_OUTPUT_FOLDER: {ADDITIONAL_OUTPUT_FOLDER}")

LYRICS_FOLDER          : C:\Users\User\OneDrive\A4\Data\Videos\AWS
ADDITIONAL_OUTPUT_FOLDER: C:\Users\User\OneDrive\A4\Data\Videos


In [2]:
# Cell 2: Load YouTube ID mapping from database
def load_youtube_mapping_from_db():
    """Load YouTube ID mapping directly from database."""
    try:
        engine = create_engine(DB_URL)
        query = """
        SELECT
            s.name AS song_title,
            CONCAT(a.first_name, ' ', a.last_name) AS artist_name,
            s.youtube_code
        FROM songs s
        JOIN artists a ON s.artist_id = a.id
        WHERE s.youtube_code IS NOT NULL AND s.youtube_code != ''
        """
        df = pd.read_sql_query(query, engine)
        df = df.rename(columns={
            'song_title':  'song_title_db',
            'artist_name': 'artist_name_db',
            'youtube_code':'youtube_code',
        })
        print(f"✅ Loaded {len(df)} YouTube IDs from database")
        return df
    except Exception as e:
        print(f"❌ Error loading YouTube mapping: {e}")
        print("💡 Check database connection and credentials")
        return pd.DataFrame(columns=['song_title_db', 'artist_name_db', 'youtube_code'])

df_mapping = load_youtube_mapping_from_db()
print(df_mapping.head())

✅ Loaded 835 YouTube IDs from database
                   song_title_db      artist_name_db youtube_code
0           Like a Rolling Stone           Bob Dylan  IwOfCgkyEj0
1  (I Can't Get No) Satisfaction  Rolling Stones The  nrIPxlFzDi0
2                        Imagine         John Lennon  VOgFZfRVaww
3                What's Going On         Marvin Gaye  H-kA3UtBj4M
4                        Respect    Aretha  Franklin  6FOUqQt3Kg0


In [3]:
# Cell 3: Filename parsing + YouTube ID lookup (with Unicode normalisation)

import re
import unicodedata

# All Unicode dash variants that must be treated as ASCII '-'
_DASH_CHARS = "\u2010\u2011\u2012\u2013\u2014\u2015\u2212"   # ‐ ‑ ‒ – — ― −

def _normalize_text(s):
    """Lowercase, unify dashes, strip punctuation, collapse whitespace."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    s = str(s)
    # 1. NFKC collapses full-width chars and some look-alikes
    s = unicodedata.normalize("NFKC", s)
    # 2. Replace every dash variant with ASCII '-'
    for ch in _DASH_CHARS:
        s = s.replace(ch, "-")
    # 3. Lowercase
    s = s.lower()
    # 4. Drop punctuation except word chars, whitespace, dash
    s = re.sub(r"[^\w\s\-]", " ", s, flags=re.UNICODE)
    # 5. Collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s


def _normalize_artist(s):
    """Like _normalize_text, but canonicalise position of leading 'the'.
    'Beatles The' -> 'the beatles',  'The Beatles' -> 'the beatles'."""
    s = _normalize_text(s)
    words = s.split()
    if len(words) >= 2 and words[-1] == "the":
        words = ["the"] + words[:-1]
    return " ".join(words)


def parse_filename(filename):
    """Split 'Song_Artist.mp4' -> (song_title, artist)."""
    name = filename[:-4]  # strip .mp4
    if '_' in name:
        parts = name.split('_')
        return parts[0].strip(), ' '.join(parts[1:]).strip()
    return name, ''


def find_youtube_id(song_title, artist, df_mapping):
    """Find YouTube ID using normalised song title and artist."""
    if df_mapping.empty:
        return None

    clean_song   = _normalize_text(song_title)
    clean_artist = _normalize_artist(artist)

    if not clean_song:
        return None

    for _, row in df_mapping.iterrows():
        db_song   = _normalize_text(row['song_title_db'])
        db_artist = _normalize_artist(row['artist_name_db'])

        # 1. Exact match on normalised values
        if db_song == clean_song and db_artist == clean_artist:
            return row['youtube_code']

        # 2. Same song title, fuzzy artist
        if db_song == clean_song and clean_artist and db_artist:
            if clean_artist in db_artist or db_artist in clean_artist:
                return row['youtube_code']

        # 3. One title contains the other, fuzzy artist
        if clean_song in db_song or db_song in clean_song:
            if clean_artist and db_artist:
                if clean_artist in db_artist or db_artist in clean_artist:
                    return row['youtube_code']

    return None

In [4]:
# Cell 4: Scan MP4 folder
def scan_mp4_files(folder_path):
    """Scan folder for MP4 files and return sorted list."""
    try:
        files = [f for f in os.listdir(folder_path) if f.lower().endswith('.mp4')]
        files.sort()
        return files
    except Exception as e:
        print(f"Error scanning folder: {e}")
        return []

mp4_files = scan_mp4_files(LYRICS_FOLDER)

if not mp4_files:
    print(f"❌ No MP4 files found in {LYRICS_FOLDER}")
else:
    print(f"✅ Found {len(mp4_files)} MP4 files:")
    for i, f in enumerate(mp4_files[:5], 1):
        print(f"   {i:2d}. {f}")
    if len(mp4_files) > 5:
        print(f"   ... and {len(mp4_files) - 5} more files ...")

✅ Found 528 MP4 files:
    1. (Everything I Do) I Do It For You_Bryan Adams.mp4
    2. (I Can't Get No) Satisfaction_Rolling Stones The.mp4
    3. (I Can't Help) Falling In Love With You_UB40.mp4
    4. (Sittin' On) The Dock of the Bay_Otis Redding.mp4
    5. 18 ฝน (TH)_เสือ ธนพล.mp4
   ... and 523 more files ...


In [5]:
# Cell 5: Build full song list with youtube_id
# This is the single source of truth used by Cells 6, 7, 8.

def build_song_records(mp4_files, df_mapping):
    records = []
    for filename in mp4_files:
        song_title, artist = parse_filename(filename)
        youtube_id = find_youtube_id(song_title, artist, df_mapping)
        records.append({
            "filename":   filename,
            "song_title": song_title,
            "artist":     artist,
            "youtube_id": youtube_id if youtube_id else "",
        })
    return records

song_records = build_song_records(mp4_files, df_mapping)

matched = sum(1 for r in song_records if r["youtube_id"])
print(f"📊 Total songs       : {len(song_records)}")
print(f"📊 With YouTube ID   : {matched}")
print(f"📊 Without YouTube ID: {len(song_records) - matched}")

📊 Total songs       : 528
📊 With YouTube ID   : 515
📊 Without YouTube ID: 13


In [6]:
# Cell 5.5 (optional): Diagnose why a song has no match
# Requires df_mapping (Cell 2). Safe to skip.

def explain_no_match(song_title, artist, df_mapping, top_n=3):
    """Print the closest DB candidates and their normalised forms."""
    clean_song   = _normalize_text(song_title)
    clean_artist = _normalize_artist(artist)

    print(f"\n🔎 Query: '{song_title}' by '{artist}'")
    print(f"   normalised song  : {clean_song!r}")
    print(f"   normalised artist: {clean_artist!r}")

    # Score candidates by how many words overlap
    scored = []
    for _, row in df_mapping.iterrows():
        db_song   = _normalize_text(row['song_title_db'])
        db_artist = _normalize_artist(row['artist_name_db'])

        s_overlap = len(set(clean_song.split())   & set(db_song.split()))
        a_overlap = len(set(clean_artist.split()) & set(db_artist.split()))
        score = s_overlap * 2 + a_overlap
        if score > 0:
            scored.append((score, db_song, db_artist, row['youtube_code']))

    scored.sort(reverse=True)

    if not scored:
        print("   ❌ No DB candidate shares even one word with this query.")
        return

    print(f"   Top {top_n} closest DB rows:")
    for score, db_song, db_artist, yt in scored[:top_n]:
        print(f"     score={score:>2}  song={db_song!r}  artist={db_artist!r}  yt={yt}")


# --- Run diagnostics on the currently-missing songs ---
# Build once with the current df_mapping to see what still fails after the fix.
_diag_records = build_song_records(mp4_files, df_mapping)
_diag_blanks  = [r for r in _diag_records if not r["youtube_id"]]

print(f"\n🚫 {len(_diag_blanks)} songs still missing after normalisation fix:")
for r in _diag_blanks:
    explain_no_match(r["song_title"], r["artist"], df_mapping)


🚫 13 songs still missing after normalisation fix:

🔎 Query: 'Back To Black' by 'Amy Winehouse'
   normalised song  : 'back to black'
   normalised artist: 'amy winehouse'
   Top 3 closest DB rows:
     score= 4  song='back in black'  artist='acdc'  yt=pAgnJDJN4VA
     score= 2  song='what s love got to do with it'  artist='tina turner'  yt=oGpFcHTxjZs
     score= 2  song='welcome to the jungle'  artist='guns n roses'  yt=o1tj2zJ2Wvg

🔎 Query: 'He Aint Heavy, Hes My Brother' by 'The Hollies'
   normalised song  : 'he aint heavy hes my brother'
   normalised artist: 'the hollies'
   Top 3 closest DB rows:
     score=10  song='he ain t heavy he s my brother'  artist='the hollies'  yt=iPDl9yKSnDE
     score= 3  song='with a little help from my friends'  artist='the beatles'  yt=0C58ttB2-Qg
     score= 3  song='while my guitar gently weeps'  artist='the beatles'  yt=VJDJs9dumZI

🔎 Query: 'I'll Never Dance Again' by 'The Wynners'
   normalised song  : 'i ll never dance again'
   normalised 

In [7]:
# Cell 6: Write songs-with-ids.json (all songs)
def write_songs_with_ids(song_records, output_file=SONGS_WITH_IDS_FILE,
                         extra_folder=None):
    matched = sum(1 for r in song_records if r["youtube_id"])
    data = {
        "generated_at":              datetime.datetime.now().isoformat(),
        "total_songs":               len(song_records),
        "songs_with_youtube_ids":    matched,
        "songs_without_youtube_ids": len(song_records) - matched,
        "songs":                     song_records,
    }

    folders = ['.']
    if extra_folder:
        folders.append(str(extra_folder))

    for folder in folders:
        try:
            path = os.path.join(folder, output_file)
            with open(path, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
            size = os.path.getsize(path) / 1024
            print(f"✅ Generated {path} ({size:.1f} KB)")
        except Exception as e:
            print(f"❌ Error generating {os.path.join(folder, output_file)}: {e}")

write_songs_with_ids(song_records, extra_folder=ADDITIONAL_OUTPUT_FOLDER)

✅ Generated .\songs-with-ids.json (97.1 KB)
✅ Generated C:\Users\User\OneDrive\A4\Data\Videos\songs-with-ids.json (97.1 KB)


In [8]:
# Cell 7: Write songs-missing-youtube-ids.json (only songs with no youtube_id)
def write_songs_missing_ids(song_records, output_file=SONGS_MISSING_IDS_FILE,
                            extra_folder=None):
    blanks = [r for r in song_records if not r["youtube_id"]]
    matched = len(song_records) - len(blanks)

    # Tag every blank song with an explicit marker
    tagged = [
        {
            "filename":      r["filename"],
            "song_title":    r["song_title"],
            "artist":        r["artist"],
            "youtube_id":    "",
            "no_youtube_id": True,
        }
        for r in blanks
    ]

    data = {
        "generated_at":              datetime.datetime.now().isoformat(),
        "total_songs":               len(song_records),
        "songs_with_youtube_ids":    matched,
        "songs_without_youtube_ids": len(blanks),
        "no_youtube_id_songs":       tagged,
    }

    folders = ['.']
    if extra_folder:
        folders.append(str(extra_folder))

    for folder in folders:
        try:
            path = os.path.join(folder, output_file)
            with open(path, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
            size = os.path.getsize(path) / 1024
            print(f"✅ Generated {path} ({size:.1f} KB)")
        except Exception as e:
            print(f"❌ Error generating {os.path.join(folder, output_file)}: {e}")

    return tagged

missing_songs = write_songs_missing_ids(
    song_records,
    extra_folder=ADDITIONAL_OUTPUT_FOLDER,
)

print(f"\n🚫 {len(missing_songs)} songs without a YouTube ID:")
for i, row in enumerate(missing_songs, 1):
    print(f"{i:4d}. ❌ '{row['song_title']}' by {row['artist']}")
    print(f"       file: {row['filename']}")

✅ Generated .\songs-missing-youtube-ids.json (3.0 KB)
✅ Generated C:\Users\User\OneDrive\A4\Data\Videos\songs-missing-youtube-ids.json (3.0 KB)

🚫 13 songs without a YouTube ID:
   1. ❌ 'Back To Black' by Amy Winehouse
       file: Back To Black_Amy Winehouse.mp4
   2. ❌ 'He Aint Heavy, Hes My Brother' by The Hollies
       file: He Aint Heavy, Hes My Brother_The Hollies.mp4
   3. ❌ 'I'll Never Dance Again' by The Wynners
       file: I'll Never Dance Again_The Wynners.mp4
   4. ❌ 'If I Cant Have You' by Yvonne Elliman
       file: If I Cant Have You_Yvonne Elliman.mp4
   5. ❌ 'In the Year 2525' by Zager & Evans
       file: In the Year 2525_Zager & Evans.mp4
   6. ❌ 'Islands In the Stream' by Dolly Parton & Kenny Rogers
       file: Islands In the Stream_Dolly Parton & Kenny Rogers.mp4
   7. ❌ 'Roses Are Red' by Bobby Vinton
       file: Roses Are Red_Bobby Vinton.mp4
   8. ❌ 'Simon Says' by 1910 Fruitgum Co
       file: Simon Says_1910 Fruitgum Co.mp4
   9. ❌ 'Stumblin' in' by Suzi Q

In [9]:
# Cell 8: Upload checklist for AWS
def generate_upload_checklist(mp4_files, lyrics_folder):
    print("\n📋 Upload Checklist for AWS:")
    print("=" * 50)

    core_files = ['index.html', 'style.css', 'script.js', SONGS_WITH_IDS_FILE]

    print("\n🚀 Core App Files:")
    for file in core_files:
        if os.path.exists(file):
            size = os.path.getsize(file) / 1024
            print(f"   ✅ {file} ({size:.1f} KB)")
        else:
            print(f"   ❌ {file} (MISSING)")

    print(f"\n🎵 MP4 Files to Upload ({len(mp4_files)} files):")
    total_size_mb = 0
    for i, mp4_file in enumerate(mp4_files[:5], 1):
        fp = os.path.join(lyrics_folder, mp4_file)
        if os.path.exists(fp):
            size = os.path.getsize(fp) / (1024 * 1024)
            total_size_mb += size
            print(f"   ✅ {i:2d}. {mp4_file} ({size:.1f} MB)")
        else:
            print(f"   ❌ {i:2d}. {mp4_file} (FILE NOT FOUND)")

    if len(mp4_files) > 5:
        print(f"   ... and {len(mp4_files) - 5} more MP4 files ...")
        for mp4_file in mp4_files[5:]:
            fp = os.path.join(lyrics_folder, mp4_file)
            if os.path.exists(fp):
                total_size_mb += os.path.getsize(fp) / (1024 * 1024)

    print(f"\n📦 Total estimated upload size: {total_size_mb:.1f} MB")
    print("\n💡 Upload all checked files to your AWS Lyrics folder")

generate_upload_checklist(mp4_files, LYRICS_FOLDER)


📋 Upload Checklist for AWS:

🚀 Core App Files:
   ❌ index.html (MISSING)
   ❌ style.css (MISSING)
   ❌ script.js (MISSING)
   ✅ songs-with-ids.json (97.1 KB)

🎵 MP4 Files to Upload (528 files):
   ✅  1. (Everything I Do) I Do It For You_Bryan Adams.mp4 (10.7 MB)
   ✅  2. (I Can't Get No) Satisfaction_Rolling Stones The.mp4 (6.7 MB)
   ✅  3. (I Can't Help) Falling In Love With You_UB40.mp4 (4.6 MB)
   ✅  4. (Sittin' On) The Dock of the Bay_Otis Redding.mp4 (3.4 MB)
   ✅  5. 18 ฝน (TH)_เสือ ธนพล.mp4 (42.6 MB)
   ... and 523 more MP4 files ...

📦 Total estimated upload size: 5941.8 MB

💡 Upload all checked files to your AWS Lyrics folder


In [10]:
# Cell 9: Final summary
print("=" * 50)
print("🎉 Script completed successfully!")
print(f"   Total songs        : {len(song_records)}")
print(f"   With YouTube ID    : {sum(1 for r in song_records if r['youtube_id'])}")
print(f"   Without YouTube ID : {len(missing_songs)}")
print(f"   songs-with-ids.json              → {SONGS_WITH_IDS_FILE}")
print(f"   songs-missing-youtube-ids.json   → {SONGS_MISSING_IDS_FILE}")
print("=" * 50)

🎉 Script completed successfully!
   Total songs        : 528
   With YouTube ID    : 515
   Without YouTube ID : 13
   songs-with-ids.json              → songs-with-ids.json
   songs-missing-youtube-ids.json   → songs-missing-youtube-ids.json
